In [ ]:
# %% [CELL 1] Configuration  <- the only cell you edit
# =============================================================================
# TRANSLATION PROBE — does translating non-Latin names help blocking?
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# Takes real TRUE pairs that blocking MISSED because the candidate name is in an
# Indian script (category "non-Latin name" in blocking_misses.csv), translates
# the candidate names to English with IndicTrans2, and measures against the
# true S1 name:
#   * how often each method makes the pair look similar (token-set >= 70 / 85)
#       - translit : our rule-based transliterated words
#       - skeleton : our phonetic skeleton (what Phase B uses)
#       - translate: IndicTrans2 English output
#       - translate_skel: skeleton of the English output
#   * false-match risk: similarity to a RANDOM other S1 name
#   * speed (names/s) and projected time for all non-Latin names
#
# NEW NOTEBOOK. Inputs: (1) the competition dataset, (2) blocking_misses.csv
# (upload it as a small dataset, or attach the miss-analysis notebook output).
# SETTINGS  GPU T4 x1, Internet ON (downloads the model, installs packages).
# RUNTIME   ~15-25 min for 3,000 names (mostly model download + translation).
# =============================================================================

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
MODEL = "ai4bharat/indictrans2-indic-en-dist-200M"
N_SAMPLE = 3_000          # missed non-Latin pairs to test
BATCH = 64
NUM_BEAMS = 1             # 1 = greedy (fast). The model card's examples use 5.
MAX_LEN = 64
SEED = 0

In [ ]:
# %% [CELL 2] Install and import
import subprocess, sys, importlib
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for mod, pkg in (("rapidfuzz", "rapidfuzz"), ("indic_transliteration", "indic_transliteration")):
    try:
        importlib.import_module(mod)
    except ImportError:
        pip(pkg)
IP_OK = False
try:
    try:
        from IndicTransToolkit.processor import IndicProcessor
    except ImportError:
        from IndicTransToolkit import IndicProcessor
    IP_OK = True
except ImportError:
    pip("IndicTransToolkit")
    try:
        try:
            from IndicTransToolkit.processor import IndicProcessor
        except ImportError:
            from IndicTransToolkit import IndicProcessor
        IP_OK = True
    except Exception as e:
        print(f"IndicTransToolkit unavailable ({e}); falling back to plain language tags.")
print("IndicProcessor available:", IP_OK)

import re, time, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
DATA_ROOT, INPUT_ROOT, WORK = Path(DATA_ROOT), Path(INPUT_ROOT), Path(WORK)
T0 = time.time()
log = lambda m: print(f"  [{(time.time()-T0)/60:4.1f} min] {m}", flush=True)

In [ ]:
# %% [CELL 3] Text helpers (rule-based baseline = what the pipeline uses now)
BLOCKS = {0x0900: ("hin_Deva", sanscript.DEVANAGARI), 0x0980: ("ben_Beng", sanscript.BENGALI),
          0x0A00: ("pan_Guru", sanscript.GURMUKHI), 0x0A80: ("guj_Gujr", sanscript.GUJARATI),
          0x0B00: ("ory_Orya", sanscript.ORIYA), 0x0B80: ("tam_Taml", sanscript.TAMIL),
          0x0C00: ("tel_Telu", sanscript.TELUGU), 0x0C80: ("kan_Knda", sanscript.KANNADA),
          0x0D00: ("mal_Mlym", sanscript.MALAYALAM)}
LEGAL = {"pvt", "private", "ltd", "limited", "llp", "llc", "lp", "pllc", "inc", "incorporated",
         "corp", "corporation", "co", "company", "pr", "pra", "li", "public"}
LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")
FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"), ("dh", "d"),
        ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"), ("z", "s"), ("w", "v"),
        ("j", "g"), ("y", ""), ("v", "b"))


def script_of(s):
    """Language tag + transliteration scheme of the first Indic character, else None."""
    for ch in str(s):
        for b, v in BLOCKS.items():
            if b <= ord(ch) < b + 0x80:
                return v
    return None


def latin_words(name):
    """Rule-based transliteration to lowercase Latin words (current pipeline)."""
    s = str(name).replace(".", "")
    s = "".join(" " if unicodedata.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        sc = script_of(w)
        if sc:
            try:
                w = transliterate(w, sc[1], sanscript.ITRANS)
            except Exception:
                pass
            if sc[0] == "tam_Taml":
                w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = unicodedata.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if sc and sc[0] == "tam_Taml":
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = re.sub(r"[^a-z0-9]", "", w)
        if sc:
            w = re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)
        if w:
            out.append(w)
    return out


def core(words):
    """Drop legal-form words so only the business name is compared."""
    return [w for w in words if w not in LEGAL and not w.startswith(LEGAL_PREFIX)]


def skeleton(words):
    """Phonetic skeleton. Order fixed vs Phase B: fold m->n BEFORE collapsing repeats."""
    s = " ".join(core(words))
    for a, b in FOLD:
        s = s.replace(a, b)
    s = re.sub(r"[aeiou]", "", s).replace("m", "n")
    return re.sub(r"\s+", " ", re.sub(r"(.)\1+", r"\1", s)).strip()


def english_words(text):
    s = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9 ]", " ", s.replace(".", "")).split()

In [ ]:
# %% [CELL 4] Data: sample of real missed non-Latin pairs
hits = sorted(INPUT_ROOT.rglob("blocking_misses.csv")) + sorted(WORK.rglob("blocking_misses.csv"))
assert hits, "blocking_misses.csv not found: attach it as a dataset"
ms = pd.read_csv(hits[0])
ms = ms[ms["category"] == "non-Latin name"]
log(f"{len(ms):,} missed non-Latin pairs in {hits[0]}")
ms = ms.sample(min(N_SAMPLE, len(ms)), random_state=SEED).reset_index(drop=True)

COLS = ["entity_id", "business_name", "business_address", "country"]
names = pd.concat([pd.read_csv(DATA_ROOT / "train" / f"train_source{i}.tsv", sep="\t", dtype=str,
                               names=COLS, header=0, keep_default_na=False, usecols=[0, 1])
                   for i in (1, 2, 3)], ignore_index=True).set_index("entity_id")["business_name"]
ms["s1_name"] = names.reindex(ms["s1"]).values
ms["cand_name"] = names.reindex(ms["cand"]).values
ms = ms.dropna(subset=["s1_name", "cand_name"]).reset_index(drop=True)
rng = np.random.default_rng(SEED)
ms["decoy_name"] = ms["s1_name"].values[rng.permutation(len(ms))]    # a random OTHER S1 name
ms["lang"] = [(script_of(n) or ("hin_Deva", None))[0] for n in ms["cand_name"]]
log(f"testing {len(ms):,} pairs; scripts: " + ", ".join(f"{k} {v}" for k, v in ms['lang'].value_counts().items()))

In [ ]:
# %% [CELL 5] Load IndicTrans2 (gated on Hugging Face: needs access + a token)
# One-time setup: (1) request access on the model page and wait for approval,
# (2) create a READ token in Hugging Face settings, (3) in Kaggle: Add-ons ->
# Secrets -> add HF_TOKEN and tick it for this notebook.
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
assert torch.cuda.is_available(), "GPU not enabled: set Accelerator to GPU T4"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, ("No Hugging Face token found: add a Kaggle secret named HF_TOKEN "
                  "(Add-ons -> Secrets) and tick it for this notebook")
try:
    from huggingface_hub import model_info
    info = model_info(MODEL, token=HF_TOKEN)
    print("model card licence:", (info.card_data or {}).get("license"),
          "| gated:", getattr(info, "gated", "unknown"))
except Exception as e:
    print("could not read the model card:", e)
t = time.time()
try:
    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, token=HF_TOKEN)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True, token=HF_TOKEN,
                                                torch_dtype=torch.float16).to("cuda").eval()
except OSError as e:
    if "gated" in str(e).lower() or "401" in str(e) or "403" in str(e):
        raise SystemExit("Access not granted yet: open the model page on Hugging Face, check that "
                         "your access request was approved, and that HF_TOKEN belongs to that account.")
    raise
ip = IndicProcessor(inference=True) if IP_OK else None
print(f"parameters: {sum(p.numel() for p in mdl.parameters())/1e6:.0f}M   loaded in {time.time()-t:.0f}s")


def translate(texts, lang):
    """Translate a list of names from `lang` to English, in batches."""
    out = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i + BATCH]
        if ip is not None:
            batch = ip.preprocess_batch(batch, src_lang=lang, tgt_lang="eng_Latn")
        else:
            batch = [f"{lang} eng_Latn {x}" for x in batch]
        enc = tok(batch, truncation=True, padding="longest", max_length=MAX_LEN,
                  return_tensors="pt").to("cuda")
        with torch.no_grad():
            try:
                gen = mdl.generate(**enc, num_beams=NUM_BEAMS, max_length=MAX_LEN, use_cache=True)
            except Exception:                        # some transformers versions: cache API mismatch
                gen = mdl.generate(**enc, num_beams=NUM_BEAMS, max_length=MAX_LEN, use_cache=False)
        dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        out += ip.postprocess_batch(dec, lang="eng_Latn") if ip is not None else dec
    return out

In [ ]:
# %% [CELL 6] Translate and time it
ms["translation"] = ""
t = time.time()
for lang, idx in ms.groupby("lang").groups.items():
    ms.loc[idx, "translation"] = translate(ms.loc[idx, "cand_name"].tolist(), lang)
secs = time.time() - t
rate = len(ms) / secs
log(f"translated {len(ms):,} names in {secs:.0f}s  ->  {rate:.0f} names/s")

In [ ]:
# %% [CELL 7] Score every method against the true S1 name (and a decoy)
def sim(a, b):
    return fuzz.token_set_ratio(a, b)

s1w = [core(english_words(n)) for n in ms["s1_name"]]
dcw = [core(english_words(n)) for n in ms["decoy_name"]]
cand_lat = [latin_words(n) for n in ms["cand_name"]]
trw = [core(english_words(n)) for n in ms["translation"]]
M = {
    "translit (rule-based words)": ([" ".join(core(w)) for w in cand_lat], [" ".join(w) for w in s1w], [" ".join(w) for w in dcw]),
    "skeleton (Phase B)": ([skeleton(w) for w in cand_lat], [skeleton(w) for w in s1w], [skeleton(w) for w in dcw]),
    "translate (IndicTrans2)": ([" ".join(w) for w in trw], [" ".join(w) for w in s1w], [" ".join(w) for w in dcw]),
    "translate -> skeleton": ([skeleton(w) for w in trw], [skeleton(w) for w in s1w], [skeleton(w) for w in dcw]),
}
S = {k: (np.array([sim(a, b) for a, b in zip(c, t)]), np.array([sim(a, b) for a, b in zip(c, d)]))
     for k, (c, t, d) in M.items()}
SHORT = {"translit (rule-based words)": "sim_translit", "skeleton (Phase B)": "sim_skeleton",
         "translate (IndicTrans2)": "sim_translate", "translate -> skeleton": "sim_translate_skel"}
for k, (tv, dv) in S.items():
    ms[SHORT[k]] = tv

print("\n" + "=" * 86)
print(f"TRUE S1 NAME vs candidate  ({len(ms):,} real missed non-Latin pairs)      "
      f"false-match risk = similarity to a RANDOM S1 name")
print("=" * 86)
print(f"  {'method':<30}{'mean':>7}{'>=70':>8}{'>=85':>8}   {'decoy >=70':>11}{'decoy >=85':>11}")
for k, (tv, dv) in S.items():
    print(f"  {k:<30}{tv.mean():7.1f}{(tv>=70).mean()*100:7.1f}%{(tv>=85).mean()*100:7.1f}%"
          f"   {(dv>=70).mean()*100:10.1f}%{(dv>=85).mean()*100:10.1f}%")

sk, tr = S["skeleton (Phase B)"][0], np.maximum(S["translate (IndicTrans2)"][0], S["translate -> skeleton"][0])
print(f"\n  translation similar (>=70) where skeleton is NOT : {((tr>=70)&(sk<70)).mean()*100:5.1f}%  <- what translation ADDS")
print(f"  skeleton similar where translation is NOT       : {((sk>=70)&(tr<70)).mean()*100:5.1f}%")
print(f"  either method similar                           : {((sk>=70)|(tr>=70)).mean()*100:5.1f}%")

print("\n  by script   (share >= 70:  skeleton | translation-best)")
for lang, g in ms.groupby("lang"):
    i = g.index.to_numpy()
    print(f"    {lang:<10}{len(i):>6}   {(sk[i]>=70).mean()*100:5.1f}% | {(tr[i]>=70).mean()*100:5.1f}%")

# projected cost for ALL non-Latin names (test pool)
t = time.time()
nl_pat = re.compile("[\u0900-\u0D7F]")
n_test = 0
for i in (2, 3):
    col = pd.read_csv(DATA_ROOT / "test" / f"test_source{i}.tsv", sep="\t", dtype=str, names=COLS,
                      header=0, keep_default_na=False, usecols=[1])["business_name"]
    n_test += int(col.str.contains(nl_pat).sum())
print(f"\n  non-Latin names in the test pool: {n_test:,}  ->  at {rate:.0f} names/s: "
      f"~{n_test/rate/60:.0f} min on one T4 (train pool similar)")

In [ ]:
# %% [CELL 8] Examples + save
def show(title, mask, n=10):
    sub = ms[mask].head(n)
    print(f"\n  --- {title}  ({int(mask.sum()):,}) ---")
    for _, r in sub.iterrows():
        print(f"   S1 : {r.s1_name}\n   CAND: {r.cand_name}\n     translation: {r.translation!r}   "
              f"skeleton sim {r['sim_skeleton']:.0f} | translation sim {r['sim_translate']:.0f} "
              f"| translation-skeleton sim {r['sim_translate_skel']:.0f}")

show("translation helps (translation >= 70, skeleton < 70)", (tr >= 70) & (sk < 70))
show("translation hurts (skeleton >= 70, translation < 70)", (sk >= 70) & (tr < 70))
show("neither works", (sk < 70) & (tr < 70))
out = WORK / "translation_probe.csv"
ms.to_csv(out, index=False)
log(f"saved {out}   DONE")